# Dip coating with rotation: grab upright, dip turning, imaging rack

The gripper points straight down and holds the electrode by the **top of its metal part**, so it hangs straight down
and turning the gripper spins it in place. After the dip the hand turns a quarter turn to lay it flat, face up, and
puts it on the right imaging rack.

**Teach once (dry electrode)**, then copy the printed numbers into cell 1 (or send them to Claude):

- **Cell 1**: settings (your dip speeds, rotation speed, the taught positions). Nothing moves.
- **Cell 2**: TEACH the grab at the stand. Prints `Grab:`.
- **Cell 3**: CHECK in the air: does it spin in place, and which way is face up. Prints `Face up:`.
- **Cell 4**: TEACH the "just above the ink" height at the usual dip location. Prints `Ink:`.
- **Cell 5**: TEACH the imaging spot for this grip (puts the dry electrode on the imaging rack). Prints `Spot:`.

**Then:**

- **Cell 6**: FULL RUN: practice the dip with the empty gripper, grab, dip with rotation, imaging rack.
- **Cell 7**: PUT BACK the electrode in its stand, if the gripper still holds it.

Usual order the first time: connect, 1, 2, 3, 4, 5 (the electrode ends on the imaging rack: put it back in its stand by
hand), then 6.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='rotation_dip_coat', connect_db=False)
arm = rt.arm

In [ ]:
# ===== 1: settings. Nothing moves in this cell. Run it first, and again after changing anything here. =====
import time

# --- the dip, with rotation (clockwise, seen from above) ---
ROTATION_SPEED_DEG_S = 10.0   # how fast the electrode turns, degrees per second (0 - 90)
DIP_SPEED_MM_S = 2.0          # going down into the ink, mm/s
DIP_DEPTH_MM = 20.0           # how far it goes down from "just above the ink" (INK_TOP_Z), mm. Must be less than
                              # the length of electrode below the fingers, so the fingers stay out of the ink.
HOLD_S = 10.0                 # how long it stays down in the ink, still turning, seconds
LIFT_SPEED_MM_S = 1.0         # coming back up out of the ink, still turning, mm/s
# It turns all the way down, while holding, and all the way up. The last joint can turn about 2 turns (700 degrees)
# in one go, so ROTATION_SPEED_DEG_S x (time down + HOLD_S + time up) must be at most 700; this cell checks it.

# --- taught positions: copy in the numbers the teach cells print ---
TAUGHT_GRAB = [154.0, 378.1, 315.0, 90.0]   # cell 2 "Grab:" x, y, z, yaw. From 2026-11-25, off-centre: teach again.
FACE_UP_YAW = 180.0                         # cell 3 "Face up:" yaw (180 or 0). 180 found on 2026-11-25.
DIP_XY = [-51.5, 257.4]                     # the usual dip location, x and y (as in all the earlier settings)
INK_TOP_Z = None                            # cell 4 "Ink:" z: the electrode's tip just above the ink. Not taught yet.
IMAGING_SPOT_ROT = None                     # cell 5 "Spot:" x, y, z on the right imaging rack. Not taught yet.
IMAGING_GUESS = [230.0, 488.6, 200.9]       # where cell 5 starts. With the old grip the spot was x = 300; this grip
                                            # holds the metal part at its end, further from the coated end, so the
                                            # spot moves left by that distance. 230 guesses 70 mm (too far left is
                                            # safe: away from the camera). Cell 5 lets you move it.

# --- after the dip: as fast as is safe, so the film does not dry on the way ---
TRAVEL_SPEED_MM_S = 100.0     # up from the beaker, across, and down beside the imaging rack
TURN_SPEED_DEG_S = 90.0       # turning the electrode face up after the dip (at most one turn)
SLIDE_SPEED_MM_S = 30.0       # low over the rack and sliding in under the camera
PLACE_SPEED_MM_S = 10.0       # the last 2.5 cm down into the slot
LIFT_OFF_SPEED_MM_S = 5.0     # lifting the open gripper off the electrode
RELEASE_STEP = 1              # letting go: the fingers open this many units at a time, all the way open,
RELEASE_STEP_PAUSE_S = 0.15   #   pausing this long after each step
CONFIRM_BEFORE_PLACING = True # stop beside the imaging rack and wait for your yes (adds drying time)

# --- heights and places ---
SAFE_Z = 500.0                # travel height, high above the bench
START_Z = 400.0               # cell 2: the open gripper stops here above the stand, for you to lower it
LOWEST_Z = 180.0              # teaching never takes the gripper lower than this
AIR_SPOT = [347.9, 308.1, 500.0]   # cell 3 checks here: high above mid_station, in open air
SLIDE_MM, SLIDE_HEIGHT_MM, HOVER_MM = 70.0, 25.0, 100.0   # imaging rack approach, as before
WIND_SPEED_DEG_S = 45.0       # winding the gripper back before the dip (dry, in the air)

cfg = arm.config
mid = list(cfg.pos_dict['mid_station'])
DOWN = [180.0, 0.0]           # roll, pitch with the gripper pointing straight down
J7_LIMIT = 350.0              # the last joint turns between -360 and +360 degrees; stay 10 inside that
# Laying the electrode flat, coated end to the right: the fingers are turned to close front-and-back (yaw 180 or
# 0), then the hand turns a quarter turn about the line between the fingers, so the gripper points right.
FLAT = {180.0: [0.0, 90.0, 0.0], 0.0: [180.0, -90.0, 0.0]}   # yaw before -> roll, pitch, yaw when flat

down_s = DIP_DEPTH_MM / DIP_SPEED_MM_S
up_s = DIP_DEPTH_MM / LIFT_SPEED_MM_S
total_turn = ROTATION_SPEED_DEG_S * (down_s + HOLD_S + up_s)
if not (0 < DIP_SPEED_MM_S <= 50 and 0 < LIFT_SPEED_MM_S <= 50 and 0 < DIP_DEPTH_MM <= 80 and HOLD_S >= 0
        and 0 <= ROTATION_SPEED_DEG_S <= 90):
    raise SystemExit('A speed, the depth or the hold time is out of range.')
if total_turn > 2 * J7_LIMIT:
    raise SystemExit(f'That is {total_turn:.0f} degrees of turning, more than the {2 * J7_LIMIT:g} the gripper can turn '
                     f'in one go. With these times the rotation speed can be at most '
                     f'{2 * J7_LIMIT / (down_s + HOLD_S + up_s):.1f} deg/s.')
if FACE_UP_YAW not in FLAT:
    raise SystemExit('FACE_UP_YAW must be 180.0 or 0.0.')
for _s in (TRAVEL_SPEED_MM_S, SLIDE_SPEED_MM_S, PLACE_SPEED_MM_S, LIFT_OFF_SPEED_MM_S):
    if not 0 < _s <= 100:
        raise SystemExit('Speeds must be between 0 and 100 mm/s.')


def at_speed(mm_s):
    """The arm's default motion settings with the speed swapped to mm_s."""
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def wrap(angle):
    """An angle in degrees, brought into -180 .. 180."""
    return (angle + 180.0) % 360.0 - 180.0


def pointing_down(pose, tol=5.0):
    return abs(wrap(pose[3] - 180.0)) <= tol and abs(pose[4]) <= tol


def lying_flat(pose, tol=5.0):
    return abs(abs(pose[4]) - 90.0) <= tol


def j7():
    """Angle of the last joint (the one that turns the gripper), in degrees."""
    return arm.get_joint_angles()[6]


def turn_j7_to(angle, speed):
    """Turn only the last joint, to angle (degrees), at speed (degrees per second). Nothing else moves."""
    if abs(angle) > J7_LIMIT + 0.5:
        raise SystemExit(f'That would turn the last joint to {angle:.0f} degrees, past its range. Stopped; tell Claude.')
    arm.set_servo_angle(servo_id=7, angle=angle, speed=speed, mvacc=200, wait=True)


def learn_turn_direction():
    """With the gripper pointing straight down, turn the last joint 5 degrees and back, to learn which way it turns."""
    global YAW_PER_J7
    a, yaw0 = j7(), arm.get_cartesian_pos()[5]
    step = -5.0 if a > 0 else 5.0
    turn_j7_to(a + step, 10)
    yaw1 = arm.get_cartesian_pos()[5]
    turn_j7_to(a, 10)
    ratio = wrap(yaw1 - yaw0) / step
    if abs(abs(ratio) - 1) > 0.3:
        raise SystemExit(f'Turning the last joint {step:+g} degrees turned the gripper {wrap(yaw1 - yaw0):+.1f} degrees; '
                         f'expected about 5. Not safe to continue. Tell Claude.')
    YAW_PER_J7 = 1.0 if ratio > 0 else -1.0


def turn_gripper_to(yaw, speed=10, centre=False):
    """
    With the gripper pointing straight down, turn it about its own axis to yaw, with the last joint only (so it stays
    exactly where it is). If the short way would pass the joint's range, it goes the long way. centre=True: of the ways
    round, take the one that leaves the last joint nearest the middle of its range.
    """
    if YAW_PER_J7 is None:
        learn_turn_direction()
    a = j7()
    target = a + wrap(yaw - arm.get_cartesian_pos()[5]) / YAW_PER_J7
    choices = [t for t in (target, target - 360, target + 360) if abs(t) <= J7_LIMIT]
    best = min(choices, key=abs) if centre else min(choices, key=lambda t: abs(t - a))
    turn_j7_to(best, speed)


def move_joints_over(target, seconds, label):
    """Move all 7 joints together to target, taking about `seconds`; then report what the gripper did."""
    now = arm.get_joint_angles()
    biggest = max(abs(t - n) for t, n in zip(target, now))
    speed = max(biggest / seconds, 0.1)
    if speed > 180:
        raise SystemExit(f'{label}: that would need {speed:.0f} deg/s, more than the arm allows. Stopped.')
    p0, a0, t0 = arm.get_cartesian_pos(), now[6], time.time()
    arm.set_servo_angle(angle=target, speed=speed, mvacc=200, wait=False)
    drift = 0.0
    while time.time() - t0 < seconds + 10:
        time.sleep(0.1)
        p = arm.get_cartesian_pos()
        drift = max(drift, ((p[0] - p0[0]) ** 2 + (p[1] - p0[1]) ** 2) ** 0.5)
        if time.time() - t0 > 0.5 and not arm.get_is_moving():
            break
    took, p1 = time.time() - t0, arm.get_cartesian_pos()
    print(f'   {label}: {took:.1f} s (planned {seconds:.1f}), turned {abs(j7() - a0):.0f} degrees, '
          f'{abs(p1[2] - p0[2]):.1f} mm in height, sideways drift at most {drift:.1f} mm')


def ask(question, answers):
    """Ask until one of the answers is typed; return it."""
    while True:
        answer = input(question).strip().lower()
        if answer in answers:
            return answer
        print('Type ' + ' or '.join(answers) + '.')


def let_go():
    """Open the fingers one small step at a time, with a pause after each, all the way open; then let it settle."""
    fully_open = int(cfg.gripper_open_dist)
    pos = int(round(arm.gripper_opening))         # about 9 when the fingers hold an electrode
    while pos < fully_open:
        pos = min(pos + int(RELEASE_STEP), fully_open)
        arm.set_gripper_position(pos, wait=True, speed=1000, auto_enable=True)
        time.sleep(RELEASE_STEP_PAUSE_S)
    time.sleep(0.5)


def gripper_has_something():
    """True if the fingers are closed on something (about 9 when they hold an electrode)."""
    return 3 <= arm.gripper_opening < 45


def need_pointing_down_high(holding_expected):
    """Stop the cell (nothing moved) unless the gripper points straight down high above the bench, holding or not."""
    here = arm.get_cartesian_pos()
    if not pointing_down(here) or here[2] < 240:
        raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                         'before this cell. The arm did not move.')
    if holding_expected and not gripper_has_something():
        raise SystemExit('The gripper is not holding the electrode. Run cell 2 first. The arm did not move.')
    if not holding_expected and gripper_has_something():
        raise SystemExit(f'The gripper seems to be holding something (opening {arm.gripper_opening:.1f}); it must be '
                         f'empty for this cell. The arm did not move.')
    return here


GRAB = globals().get('GRAB')                # set by cell 2 in this session (otherwise TAUGHT_GRAB is used)
YAW_PER_J7 = globals().get('YAW_PER_J7')    # +1 or -1: how the gripper turns when the last joint turns (learned)
if INK_TOP_Z is None and globals().get('ink_taught') is not None:
    DIP_XY[0], DIP_XY[1], INK_TOP_Z = ink_taught     # taught by cell 4 in this session
if IMAGING_SPOT_ROT is None and globals().get('spot_taught') is not None:
    IMAGING_SPOT_ROT = spot_taught                   # taught by cell 5 in this session


def grab_pose():
    g = GRAB if GRAB is not None else [TAUGHT_GRAB[0], TAUGHT_GRAB[1], TAUGHT_GRAB[2], *DOWN, TAUGHT_GRAB[3]]
    return [float(v) for v in g]


print(f'Settings loaded. Dip: down {DIP_DEPTH_MM:g} mm at {DIP_SPEED_MM_S:g} mm/s, hold {HOLD_S:g} s, up at '
      f'{LIFT_SPEED_MM_S:g} mm/s, turning {ROTATION_SPEED_DEG_S:g} deg/s clockwise: {total_turn:.0f} degrees in all.')
print('Still to teach: ' + (', '.join(n for n, v in (('the ink height (cell 4)', INK_TOP_Z),
                                                     ('the imaging spot (cell 5)', IMAGING_SPOT_ROT)) if v is None)
                            or 'nothing') + '. Nothing has moved.')

In [ ]:
# ===== 2: TEACH the grab. Gripper EMPTY; the electrode standing upright in its stand, metal part up, dry. =====
# The arm goes straight up, across to above the stand (TAUGHT_GRAB), opens the gripper and comes down to START_Z.
# Then you move it with typed commands (directions as seen standing in front of the bench, facing the wall):
#   d 10 / u 10   -> 10 mm lower / higher (at most 20 mm at a time; use 1-2 mm when close)
#   l 2 / r 2     -> 2 mm left / right      f 2 / b 2 -> 2 mm towards you / towards the wall (at most 5 mm)
#   t 10 / t -10  -> turn the gripper 10 degrees clockwise / counterclockwise, seen from above (at most 90)
#   grab          -> close the fingers here
#   open          -> open the fingers again (only where it grabbed, so nothing falls)
#   done          -> after grab: lift it straight up out of its stand, slowly, and print the grab for Claude
#   back          -> give up (fingers open): go straight up
# Aim: the fingers come down around the TOP of the metal part and close on its two narrow sides (the same sides they
# hold today), as deep as the metal part allows without touching the electrode itself. The metal part must be in the
# MIDDLE of the fingers, side to side AND front to back, or the electrode swings round in a circle instead of
# spinning in place. Turn (t) only while the fingers are still above the metal part. Once it is grabbed, only u, d,
# open and done work (no sideways moves, so the stand is not pulled).
here = need_pointing_down_high(holding_expected=False)
SX, SY, GRAB_YAW = TAUGHT_GRAB[0], TAUGHT_GRAB[1], TAUGHT_GRAB[3]
print(f'TEACH the grab. The arm will go straight up to z = {SAFE_Z:g}, across to above the stand (x = {SX:.1f}, '
      f'y = {SY:.1f}), open the gripper, and come down to z = {START_Z:g}.')
if input('Gripper empty, electrode standing upright in its stand (metal part up), nothing else under the gripper, '
         'hand on the e-stop? Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> straight up')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
print('-> across, to above the stand')
arm.move_to_pos([SX, SY, SAFE_Z, *here[3:]], **at_speed(30))
learn_turn_direction()
turn_gripper_to(GRAB_YAW, speed=30)
arm.open_gripper()
print(f'-> down to z = {START_Z:g}')
arm.move_to_pos([SX, SY, START_Z, *DOWN, GRAB_YAW], **at_speed(20))

MOVES = {'l': (-1, 0), 'r': (1, 0), 'f': (0, -1), 'b': (0, 1)}
tx, ty, tz, tyaw = SX, SY, START_Z, GRAB_YAW
turned = 0.0              # degrees turned clockwise (seen from above) from TAUGHT_GRAB's yaw
holding, grab_at = False, None
print('Ready. Lower the gripper (d) until the fingers are around the top of the metal part, in the middle; then grab.')
while True:
    cmd = input('d/u, l/r/f/b + mm, t + degrees, or grab, open, done, back: ').strip().lower().split()
    if not cmd:
        continue
    word = cmd[0]
    if cmd == ['grab']:
        if holding:
            print('Already holding it. Type done to lift it straight up out of its stand.')
            continue
        arm.close_gripper()
        if arm.gripper_opening < 3:
            print(f'The fingers closed on nothing (opening {arm.gripper_opening:.1f}). Opening again; move and try again.')
            arm.open_gripper()
            continue
        holding, grab_at = True, [tx, ty, tz, *DOWN, tyaw]
        print(f'Holding it (opening {arm.gripper_opening:.1f}). Type done to lift it straight up out of its stand.')
        continue
    if cmd == ['open']:
        if holding and tz > grab_at[2] + 0.5:
            print('It would fall. Lower it back into its stand first (d), then type open.')
            continue
        arm.open_gripper()
        holding, grab_at = False, None
        print('Fingers open.')
        continue
    if cmd == ['done']:
        if not holding:
            print('Grab it first.')
            continue
        if ask('Lift it straight up out of its stand, slowly? Type yes, or no to keep adjusting: ', ('yes', 'no')) == 'yes':
            break
        continue
    if cmd == ['back']:
        if holding:
            print('Still holding it. Lower it back into its stand (d), type open, then back.')
            continue
        break
    if len(cmd) != 2 or word not in ('d', 'u', 'l', 'r', 'f', 'b', 't'):
        print('Did not understand. Examples: d 10, d 1, u 5, l 2, f 1, t 10, t -5, grab, open, done, back.')
        continue
    try:
        n = float(cmd[1])
    except ValueError:
        print('The second part must be a number, e.g. d 5.')
        continue
    if word == 't':
        if holding:
            print('Not while holding it. (To turn the grab: lower it back, open, lift a little, then turn.)')
            continue
        if not 0 < abs(n) <= 90 or abs(turned + n) > 180:
            print('Use an angle between -90 and 90 (not 0), at most half a turn in all.')
            continue
        turned += n
        tyaw = wrap(tyaw - n)           # clockwise seen from above = yaw going down
        turn_gripper_to(tyaw)
        print(f'Turned. Now {turned:+g} degrees from TAUGHT_GRAB\'s yaw.')
        continue
    nx, ny, nz = tx, ty, tz
    if word in ('d', 'u'):
        if not 0 < n <= 20:
            print('Use a distance between 0 and 20 mm.')
            continue
        nz = tz - n if word == 'd' else tz + n
        if not LOWEST_Z <= nz <= SAFE_Z:
            print(f'That is outside {LOWEST_Z:g} - {SAFE_Z:g} mm in height, so the arm did not move.')
            continue
        if holding and nz < grab_at[2] - 0.01:
            print('That would push the electrode down into its stand, so the arm did not move.')
            continue
        speed = 5 if holding else 10
    else:
        if holding:
            print('Not while holding it (it would pull on the stand). Type done to lift it out.')
            continue
        if not 0 < n <= 5:
            print('Use a distance between 0 and 5 mm.')
            continue
        dx, dy = MOVES[word]
        nx, ny = tx + dx * n, ty + dy * n
        if max(abs(nx - SX), abs(ny - SY)) > 30:
            print('That would be more than 30 mm from TAUGHT_GRAB, so the arm did not move. Tell Claude.')
            continue
        speed = 5
    tx, ty, tz = nx, ny, nz
    arm.move_to_pos([tx, ty, tz, *DOWN, tyaw], **at_speed(speed))
    print(f'Moved. Now x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}'
          + (f'; {tz - grab_at[2]:.1f} mm above where it grabbed.' if holding else '.'))

print('-> straight up' + (', slowly, out of its stand' if holding else ''))
if holding:
    arm.move_to_pos([tx, ty, min(tz + 60, SAFE_Z), *DOWN, tyaw], **at_speed(10))
arm.move_to_pos([tx, ty, SAFE_Z, *DOWN, tyaw], **at_speed(30))
if holding:
    GRAB = grab_at
    print(f'Grab: x = {GRAB[0]:.1f}, y = {GRAB[1]:.1f}, z = {GRAB[2]:.1f}, yaw = {GRAB[5]:.1f}   <- send this line to Claude')
    print('Holding the electrode high above its stand. Next: cell 3, 4 or 5, or cell 7 to put it back.')
else:
    print('Nothing grabbed; the gripper is open and high above the bench.')

In [ ]:
# ===== 3: CHECK in the air, holding the electrode (after cell 2). Nothing touches anything. =====
# High above mid_station:
#   a) a half turn and back: does the electrode hang straight and spin in place, or does its tip swing round?
#   b) the quarter turn of the hand that lays it flat for the imaging rack, coated end to the right: is it face up
#      and level? If it comes out face down, type flip to try the other way.
# Then it turns back to pointing straight down, still holding the electrode.
here = need_pointing_down_high(holding_expected=True)
if here[2] < SAFE_Z - 5:
    raise SystemExit(f'Start this cell high, at z = {SAFE_Z:g}, where cell 2 leaves it. The arm did not move.')
if input('Holding the electrode, nothing near the space high above mid_station, hand on the e-stop? '
         'Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> across, to high above mid_station')
arm.move_to_pos([*AIR_SPOT, *here[3:]], **at_speed(30))

print('a) turning the electrode 180 degrees and back, at 30 deg/s: watch its bottom tip')
a = j7()
turn_j7_to(a - 180 if a > 0 else a + 180, 30)
turn_j7_to(a, 30)
spins = ask('Did it hang straight down and spin in place (its tip did not swing round in a circle)? yes / no: ',
            ('yes', 'no'))

pre, face = FACE_UP_YAW, None
for attempt in range(3):
    print('b) turning the fingers front-and-back, then laying the electrode flat, coated end to the right')
    turn_gripper_to(pre, speed=30, centre=True)
    arm.move_to_pos([*AIR_SPOT, *FLAT[pre]], **at_speed(20))
    answer = ask('It should be lying flat, pointing to the right. Type yes (face UP and level), flip (face DOWN), '
                 'edge (on its edge), or no (something else is wrong): ', ('yes', 'flip', 'edge', 'no'))
    print('   back to pointing straight down')
    arm.move_to_pos([*AIR_SPOT, *DOWN, pre], **at_speed(20))
    if answer != 'flip':
        face = answer
        break
    pre = 0.0 if pre == 180.0 else 180.0

if face == 'yes':
    FACE_UP_YAW = pre
    print(f'Face up: yaw {pre:g}; spins in place: {spins}   <- send this line to Claude')
elif face == 'edge':
    print('It lay on its edge: the fingers hold the metal part by its flat faces, not its narrow sides. Put it back '
          '(cell 7) and teach the grab again (cell 2), turning the gripper a quarter turn (t 90) before grabbing.')
else:
    print(f'Not right yet (spins in place: {spins}). Tell Claude what you saw.')
if spins == 'no':
    print('It did not spin in place: most likely the metal part is not in the middle of the fingers (front to back or '
          'side to side), or it hangs tilted. Put it back (cell 7) and teach the grab again (cell 2), centring it.')
print('Pointing straight down, holding the electrode, high above mid_station. Next: cell 4, 5, or 7.')

In [ ]:
# ===== 4: TEACH the "just above the ink" height at the dip location, holding the electrode (after cell 2) =====
# The beaker with ink at its usual place and level. The arm goes high across to above the dip location (DIP_XY) and
# comes down to 50 mm below SAFE_Z. Then you lower it until the electrode's bottom tip is 1-2 mm ABOVE the ink,
# not touching. That is where the rotation will start, and the electrode then goes DIP_DEPTH_MM further down.
#   d 10 / u 10   -> 10 mm lower / higher (at most 20 mm at a time; use 1 mm close to the ink)
#   l 1 / r 1 / f 1 / b 1 -> 1 mm sideways, to centre it over the beaker (at most 3 mm at a time, 20 mm in all)
#   ok            -> this is the height: remember it (and x, y), go back up, print it for Claude
#   back          -> go back up without remembering anything
here = need_pointing_down_high(holding_expected=True)
print(f'TEACH the ink height. The arm will go straight up to z = {SAFE_Z:g}, across to above the dip location '
      f'(x = {DIP_XY[0]:.1f}, y = {DIP_XY[1]:.1f}), and come down to z = {SAFE_Z - 50:g}.')
if input('Holding a DRY electrode, beaker with ink in its usual place, nothing above it, hand on the e-stop? '
         'Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> straight up, then across to above the dip location')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
arm.move_to_pos([DIP_XY[0], DIP_XY[1], SAFE_Z, *here[3:]], **at_speed(30))
rpy = arm.get_cartesian_pos()[3:]
tx, ty, tz = DIP_XY[0], DIP_XY[1], SAFE_Z - 50
arm.move_to_pos([tx, ty, tz, *rpy], **at_speed(20))
MOVES = {'l': (-1, 0), 'r': (1, 0), 'f': (0, -1), 'b': (0, 1)}
found = False
while True:
    cmd = input('d/u + mm, l/r/f/b + mm, or ok, back: ').strip().lower().split()
    if not cmd:
        continue
    if cmd == ['ok']:
        if ask(f'Electrode tip 1-2 mm above the ink, not touching it, at z = {tz:.1f}? Type yes or no: ',
               ('yes', 'no')) == 'yes':
            found = True
            break
        continue
    if cmd == ['back']:
        break
    if len(cmd) != 2 or cmd[0] not in ('d', 'u', 'l', 'r', 'f', 'b'):
        print('Did not understand. Examples: d 10, d 1, u 2, l 1, b 1, ok, back.')
        continue
    try:
        n = float(cmd[1])
    except ValueError:
        print('The second part must be a number, e.g. d 5.')
        continue
    nx, ny, nz = tx, ty, tz
    if cmd[0] in ('d', 'u'):
        if not 0 < n <= 20:
            print('Use a distance between 0 and 20 mm.')
            continue
        nz = tz - n if cmd[0] == 'd' else tz + n
        if not LOWEST_Z <= nz <= SAFE_Z:
            print(f'That is outside {LOWEST_Z:g} - {SAFE_Z:g} mm in height, so the arm did not move. Tell Claude.')
            continue
        speed = 5
    else:
        if not 0 < n <= 3:
            print('Use a distance between 0 and 3 mm.')
            continue
        dx, dy = MOVES[cmd[0]]
        nx, ny = tx + dx * n, ty + dy * n
        if max(abs(nx - DIP_XY[0]), abs(ny - DIP_XY[1])) > 20:
            print('That would be more than 20 mm from the usual dip location, so the arm did not move. Tell Claude.')
            continue
        speed = 3
    tx, ty, tz = nx, ny, nz
    arm.move_to_pos([tx, ty, tz, *rpy], **at_speed(speed))
    print(f'Moved. Now x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}.')

print('-> straight up')
arm.move_to_pos([tx, ty, SAFE_Z, *rpy], **at_speed(20))
if found:
    ink_taught = [tx, ty, tz]
    DIP_XY, INK_TOP_Z = [tx, ty], tz
    print(f'Ink: x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}   <- send this line to Claude')
    print(f'The full run will start turning at z = {tz:.1f} and go down to z = {tz - DIP_DEPTH_MM:.1f} '
          f'(DIP_DEPTH_MM = {DIP_DEPTH_MM:g}). Still holding the electrode, high above the beaker.')
else:
    print('Nothing remembered. Still holding the electrode, high above the beaker.')

In [ ]:
# ===== 5: TEACH the imaging spot for this grip, holding a DRY electrode (after cell 2). The imaging slot empty. =====
# High up, the arm turns the electrode face up and lays it flat (coated end to the right), goes across to beside the
# right imaging rack, comes down beside the spot (it asks first), slides in under the camera and holds the electrode
# 10 mm above the slot. With this grip the gripper lies on its side to the LEFT of the electrode, at the rack's
# height, with the wrist about 20 cm further left: watch that nothing is in its way.
# Then you move it until it is exactly where you want it, lower it (d 1) until it just rests on the rack, and let go:
#   l 5 / r 5     -> 5 mm left / right (at most 10 mm at a time)
#   f 2 / b 2     -> 2 mm towards you / towards the wall (at most 5 mm; go up a little first, u 3, to clear the teeth)
#   d 1 / u 1     -> 1 mm lower / higher (at most 3 mm)
#   release       -> let go here, slowly (asks first), print the spot for Claude, then move away
#   back          -> give up: lift, slide out, go up, still holding the electrode (then cell 7 puts it back)
here = need_pointing_down_high(holding_expected=True)
plan = [float(v) for v in (IMAGING_SPOT_ROT if IMAGING_SPOT_ROT is not None else IMAGING_GUESS)]
flat = FLAT[FACE_UP_YAW]
side_x = plan[0] - SLIDE_MM
print(f'TEACH the imaging spot for this grip, starting from x = {plan[0]:.1f}, y = {plan[1]:.1f}, z = {plan[2]:.1f}.')
if input('Holding a DRY electrode, imaging slot empty, camera lens at least 5 cm above the rack, hand on the e-stop? '
         'Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> straight up, turning it face up')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
turn_gripper_to(FACE_UP_YAW, speed=30, centre=True)
print('-> across, laying it flat, to beside the imaging rack')
arm.move_to_pos([side_x, plan[1], SAFE_Z, *flat], **at_speed(30))
print('-> down, beside the imaging spot')
arm.move_to_pos([side_x, plan[1], plan[2] + HOVER_MM, *flat], **at_speed(30))
if ask('Electrode lying flat and face up, coated end to the right, nothing in the way of the gripper or of sliding in '
       'under the camera? Type yes to slide in, or back: ', ('yes', 'back')) == 'back':
    arm.move_to_pos([side_x, plan[1], SAFE_Z, *flat], **at_speed(30))
    arm.move_to_pos([side_x, plan[1], SAFE_Z, *DOWN, FACE_UP_YAW], **at_speed(20))
    raise SystemExit('Back up high, pointing straight down, still holding the electrode. Cell 7 puts it back.')
print('-> down low over the rack, then sliding in under the camera, to 10 mm above the slot')
arm.move_to_pos([side_x, plan[1], plan[2] + SLIDE_HEIGHT_MM, *flat], **at_speed(20))
arm.move_to_pos([plan[0], plan[1], plan[2] + SLIDE_HEIGHT_MM, *flat], **at_speed(20))
tx, ty, tz = plan[0], plan[1], plan[2] + 10
arm.move_to_pos([tx, ty, tz, *flat], **at_speed(5))
MOVES = {'l': (-1, 0), 'r': (1, 0), 'f': (0, -1), 'b': (0, 1)}
print('Holding it 10 mm above the slot. Move it until it is where you want it, lower it (d 1) until it just rests on '
      'the rack, then type release.')
released = False
while True:
    cmd = input('l/r/f/b or d/u + mm, or release, back: ').strip().lower().split()
    if not cmd:
        continue
    if cmd == ['release']:
        if ask('Electrode exactly where you want it, resting on the rack? Type yes to let go, or no: ',
               ('yes', 'no')) == 'yes':
            let_go()
            released = True
            break
        continue
    if cmd == ['back']:
        break
    if len(cmd) != 2 or cmd[0] not in ('l', 'r', 'f', 'b', 'd', 'u'):
        print('Did not understand. Examples: r 5, l 2, f 1, b 1, d 1, u 3, release, back.')
        continue
    try:
        n = float(cmd[1])
    except ValueError:
        print('The second part must be a number of mm, e.g. r 5.')
        continue
    nx, ny, nz = tx, ty, tz
    if cmd[0] in ('d', 'u'):
        if not 0 < n <= 3:
            print('Use a distance between 0 and 3 mm.')
            continue
        nz = tz - n if cmd[0] == 'd' else tz + n
        if not plan[2] - 8 <= nz <= plan[2] + SLIDE_HEIGHT_MM:
            print('That is outside the allowed height, so the arm did not move. Tell Claude.')
            continue
    else:
        limit = 10 if cmd[0] in ('l', 'r') else 5
        if not 0 < n <= limit:
            print(f'Use a distance between 0 and {limit} mm.')
            continue
        dx, dy = MOVES[cmd[0]]
        nx, ny = tx + dx * n, ty + dy * n
        if abs(nx - plan[0]) > 90 or abs(ny - plan[1]) > 20:
            print('That would be too far from where it started (90 mm left/right, 20 mm front/back), so the arm did '
                  'not move. Tell Claude.')
            continue
    tx, ty, tz = nx, ny, nz
    arm.move_to_pos([tx, ty, tz, *flat], **at_speed(3))
    print(f'Moved. Now x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f} ({tz - plan[2]:+.1f} mm from the planned height).')

if released:
    spot_taught = [tx, ty, tz]
    IMAGING_SPOT_ROT = spot_taught
    print(f'Spot: x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}   <- send this line to Claude')
    input('Electrode let go. Look at it; then press Enter (or OK) to move the arm away: up, slide out, up high. ')
    print('-> lifting off, then sliding out, away from the camera')
    arm.move_to_pos([tx, ty, tz + SLIDE_HEIGHT_MM, *flat], **at_speed(LIFT_OFF_SPEED_MM_S))
else:
    print('Giving up: lifting, sliding out, still holding the electrode.')
    arm.move_to_pos([tx, ty, max(tz, plan[2]) + SLIDE_HEIGHT_MM, *flat], **at_speed(5))
cur = arm.get_cartesian_pos()
arm.move_to_pos([tx - SLIDE_MM, ty, cur[2], *flat], **at_speed(20))
print('-> up high, then back to pointing straight down')
arm.move_to_pos([tx - SLIDE_MM, ty, SAFE_Z, *flat], **at_speed(30))
arm.move_to_pos([tx - SLIDE_MM, ty, SAFE_Z, *DOWN, FACE_UP_YAW], **at_speed(20))
if released:
    print('-> to mid_station')
    arm.move_to_pos([mid[0], mid[1], SAFE_Z, *mid[3:]], **at_speed(30))
    arm.move_to_pos(mid, **at_speed(30))
    print('Done. The electrode is on the imaging rack; the gripper is empty, at mid_station.')
else:
    print('High beside the imaging rack, pointing straight down, still holding the electrode. Cell 7 puts it back.')

In [ ]:
# ===== 6: FULL RUN: grab -> dip with rotation at the dip location -> imaging rack =====
# Gripper EMPTY at the start; electrode standing in its stand; beaker with ink in place; imaging slot empty.
#  [1] practice, with the EMPTY gripper: over the dip location, wind the gripper to its counterclockwise end, go down
#      to the ink height and DIP_DEPTH_MM further, and back up. This teaches the arm's joint angles for the dip
#      (the fingers stay where they will be in the real dip: above the ink). Nothing is dipped.
#  [2] grab the electrode from its stand and lift it out.
#  [3] back over the dip location, the arm in exactly the practised position, down to just above the ink.
#  [4] the dip: down DIP_DEPTH_MM turning, hold turning, up turning (speeds and times from cell 1).
#  [5] fast: up, turn it face up, across laying it flat, down beside the imaging rack (asks first if
#      CONFIRM_BEFORE_PLACING), slide in under the camera, put it down, let go slowly; back to mid_station.
if INK_TOP_Z is None or IMAGING_SPOT_ROT is None:
    raise SystemExit('Teach the ' + ' and the '.join(n for n, v in (('ink height (cell 4)', INK_TOP_Z),
                     ('imaging spot (cell 5)', IMAGING_SPOT_ROT)) if v is None) + ' first. The arm did not move.')
here = need_pointing_down_high(holding_expected=False)
g = grab_pose()
ix, iy, iz = (float(v) for v in IMAGING_SPOT_ROT)
flat, side_x = FLAT[FACE_UP_YAW], ix - SLIDE_MM
print(f'FULL RUN. Grab at {[round(v, 1) for v in g[:3]]}; dip at x = {DIP_XY[0]:.1f}, y = {DIP_XY[1]:.1f}: turning from '
      f'z = {INK_TOP_Z:.1f} down to {INK_TOP_Z - DIP_DEPTH_MM:.1f} at {DIP_SPEED_MM_S:g} mm/s, hold {HOLD_S:g} s, up at '
      f'{LIFT_SPEED_MM_S:g} mm/s, {ROTATION_SPEED_DEG_S:g} deg/s clockwise ({total_turn:.0f} degrees); then the imaging '
      f'rack at x = {ix:.1f}, y = {iy:.1f}, z = {iz:.1f}.')
if input('Gripper empty, electrode in its stand, beaker with ink in place, imaging slot empty, hand on the e-stop? '
         'Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()

print('[1] practice over the dip location, with the EMPTY gripper')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
arm.move_to_pos([DIP_XY[0], DIP_XY[1], SAFE_Z, *here[3:]], **at_speed(50))
if YAW_PER_J7 is None:
    learn_turn_direction()
cw = -YAW_PER_J7                       # last joint's direction for clockwise, seen from above
turn_j7_to(-cw * J7_LIMIT, WIND_SPEED_DEG_S)        # wound to the counterclockwise end
high_joints, high_pose = arm.get_joint_angles(), arm.get_cartesian_pos()
arm.move_to_pos([DIP_XY[0], DIP_XY[1], INK_TOP_Z, *high_pose[3:]], **at_speed(30))
top = arm.get_joint_angles()
arm.move_to_pos([DIP_XY[0], DIP_XY[1], INK_TOP_Z - DIP_DEPTH_MM, *high_pose[3:]], **at_speed(10))
bottom = arm.get_joint_angles()
arm.move_to_pos(high_pose, **at_speed(30))
if abs(bottom[6] - top[6]) > 1:
    raise SystemExit('The last joint turned while going straight down in the practice; not safe. Tell Claude.')

print('[2] grab the electrode from its stand')
arm.move_to_pos([g[0], g[1], SAFE_Z, *high_pose[3:]], **at_speed(50))
turn_gripper_to(g[5], speed=60)
arm.open_gripper()
arm.move_to_pos([g[0], g[1], g[2] + 40, *g[3:]], **at_speed(30))
arm.move_to_pos(g, **at_speed(10))
arm.close_gripper()
opening = arm.gripper_opening
if opening < 3:
    arm.open_gripper()
    arm.move_to_pos([g[0], g[1], SAFE_Z, *g[3:]], **at_speed(20))
    raise SystemExit(f'The fingers closed on nothing (opening {opening:.1f}). Nothing was dipped. '
                     'Check the electrode in its stand, or teach the grab again (cell 2).')
arm.move_to_pos([g[0], g[1], min(g[2] + 60, SAFE_Z), *g[3:]], **at_speed(10))
arm.move_to_pos([g[0], g[1], SAFE_Z, *g[3:]], **at_speed(30))

print('[3] over the dip location, the arm exactly as practised, then down to just above the ink')
arm.move_to_pos([DIP_XY[0], DIP_XY[1], SAFE_Z, *g[3:]], **at_speed(50))
arm.set_servo_angle(angle=high_joints, speed=WIND_SPEED_DEG_S, mvacc=200, wait=True)
p = arm.get_cartesian_pos()
if max(abs(a - b) for a, b in zip(p[:3], high_pose[:3])) > 2:
    raise SystemExit('The arm is not where it practised, so it did not go down. Still holding the electrode, high '
                     'above the beaker; cell 7 puts it back. Tell Claude.')
arm.move_to_pos([DIP_XY[0], DIP_XY[1], INK_TOP_Z, *high_pose[3:]], **at_speed(30))
off = max(abs(a - b) for a, b in zip(arm.get_joint_angles(), top))
if off > 1:
    arm.move_to_pos(high_pose, **at_speed(30))
    raise SystemExit(f'The arm came down differently from the practice (joints off by up to {off:.1f} degrees), so '
                     'it did not dip. Still holding the electrode, high above the beaker; cell 7 puts it back. Tell Claude.')

w = ROTATION_SPEED_DEG_S
print(f'[4] the dip: down {DIP_DEPTH_MM:g} mm turning, hold {HOLD_S:g} s turning, up turning')
move_joints_over(bottom[:6] + [j7() + cw * w * down_s], down_s, 'down into the ink')
arm.sample_in_flask = True
if HOLD_S > 0:
    t0, a0 = time.time(), j7()
    if w > 0:
        turn_j7_to(a0 + cw * w * HOLD_S, w)
    else:
        time.sleep(HOLD_S)
    print(f'   hold: {time.time() - t0:.1f} s (planned {HOLD_S:g}), turned {abs(j7() - a0):.0f} degrees')
move_joints_over(top[:6] + [j7() + cw * w * up_s], up_s, 'up out of the ink')
arm.sample_in_flask = False
t_out = time.time()

print(f'[5] to the imaging rack, fast')
arm.move_to_pos([DIP_XY[0], DIP_XY[1], SAFE_Z, *arm.get_cartesian_pos()[3:]], **at_speed(TRAVEL_SPEED_MM_S))
turn_gripper_to(FACE_UP_YAW, speed=TURN_SPEED_DEG_S, centre=True)
arm.move_to_pos([side_x, iy, SAFE_Z, *flat], **at_speed(TRAVEL_SPEED_MM_S))      # across, laying it flat
arm.move_to_pos([side_x, iy, iz + HOVER_MM, *flat], **at_speed(TRAVEL_SPEED_MM_S))
if CONFIRM_BEFORE_PLACING and ask('Beside the imaging rack, face up, nothing in the way? Type yes to put it down, '
                                  'or stop: ', ('yes', 'stop')) == 'stop':
    arm.move_to_pos([side_x, iy, SAFE_Z, *flat], **at_speed(30))
    arm.move_to_pos([side_x, iy, SAFE_Z, *DOWN, FACE_UP_YAW], **at_speed(20))
    raise SystemExit('Stopped high beside the imaging rack, pointing straight down, still holding the electrode. '
                     'Cell 7 puts it back in its stand.')
arm.move_to_pos([side_x, iy, iz + SLIDE_HEIGHT_MM, *flat], **at_speed(SLIDE_SPEED_MM_S))
arm.move_to_pos([ix, iy, iz + SLIDE_HEIGHT_MM, *flat], **at_speed(SLIDE_SPEED_MM_S))
arm.move_to_pos([ix, iy, iz, *flat], **at_speed(PLACE_SPEED_MM_S))
print(f'   on the imaging rack {time.time() - t_out:.0f} s after leaving the ink; letting go slowly')
let_go()
arm.move_to_pos([ix, iy, iz + SLIDE_HEIGHT_MM, *flat], **at_speed(LIFT_OFF_SPEED_MM_S))
arm.move_to_pos([side_x, iy, iz + SLIDE_HEIGHT_MM, *flat], **at_speed(SLIDE_SPEED_MM_S))
arm.move_to_pos([side_x, iy, SAFE_Z, *flat], **at_speed(TRAVEL_SPEED_MM_S))
arm.move_to_pos([side_x, iy, SAFE_Z, *DOWN, FACE_UP_YAW], **at_speed(30))
arm.move_to_pos([mid[0], mid[1], SAFE_Z, *mid[3:]], **at_speed(TRAVEL_SPEED_MM_S))
arm.move_to_pos(mid, **at_speed(50))
print('Done. The electrode is on the imaging rack; the gripper is empty, at mid_station.')

In [ ]:
# ===== 7: PUT BACK the electrode in its stand (if the gripper still holds it), let go slowly, and go up =====
# The arm takes it high across to above the stand and stops PUT_BACK_ABOVE_MM above where it grabbed. Then you lower it
# into the stand with typed commands; it never goes lower than where it grabbed:
#   d 10 / u 10   -> 10 mm lower / higher (at most 20 mm at a time; use 1-2 mm when close)
#   l 1 / r 1 / f 1 / b 1 -> 1 mm sideways to line it up (at most 3 mm at a time, 10 mm in all)
#   release       -> let go here, slowly, then go up
#   stop          -> stop here, still holding it (run this cell again to carry on)
PUT_BACK_ABOVE_MM = 80.0
here = arm.get_cartesian_pos()
if not gripper_has_something():
    raise SystemExit('The gripper is not holding anything, so there is nothing to put back. The arm did not move.')
if lying_flat(here) and here[2] < SAFE_Z - 5:
    raise SystemExit('The electrode is lying flat low down (near the imaging rack?). Tell Claude. The arm did not move.')
if not lying_flat(here) and (not pointing_down(here) or here[2] < 240):
    raise SystemExit('The gripper must point straight down at least 5 cm above the racks (z >= 240). Tell Claude. '
                     'The arm did not move.')
g = grab_pose()
if input('Stand empty and in place, hand on the e-stop? Type yes to take the electrode back to its stand: '
         ).strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
if lying_flat(here):
    print('-> back to pointing straight down')
    arm.move_to_pos([*here[:3], *DOWN, 180.0 if here[4] > 0 else 0.0], **at_speed(20))
    here = arm.get_cartesian_pos()
print('-> up and across, high above the bench, to above the stand')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
arm.move_to_pos([g[0], g[1], SAFE_Z, *here[3:]], **at_speed(30))
turn_gripper_to(g[5], speed=30)
tx, ty, tz = g[0], g[1], min(g[2] + PUT_BACK_ABOVE_MM, SAFE_Z)
print(f'-> down to {tz - g[2]:.0f} mm above where it grabbed')
arm.move_to_pos([tx, ty, tz, *g[3:]], **at_speed(20))
MOVES = {'l': (-1, 0), 'r': (1, 0), 'f': (0, -1), 'b': (0, 1)}
released = False
while True:
    cmd = input('d/u + mm, l/r/f/b + mm, or release, stop: ').strip().lower().split()
    if not cmd:
        continue
    if cmd == ['release']:
        if ask('Electrode in its stand, standing by itself? Type yes to let go, or no: ', ('yes', 'no')) == 'yes':
            let_go()
            released = True
            break
        continue
    if cmd == ['stop']:
        break
    if len(cmd) != 2 or cmd[0] not in ('d', 'u', 'l', 'r', 'f', 'b'):
        print('Did not understand. Examples: d 10, d 1, u 2, l 1, release, stop.')
        continue
    try:
        n = float(cmd[1])
    except ValueError:
        print('The second part must be a number, e.g. d 5.')
        continue
    nx, ny, nz = tx, ty, tz
    if cmd[0] in ('d', 'u'):
        if not 0 < n <= 20:
            print('Use a distance between 0 and 20 mm.')
            continue
        nz = tz - n if cmd[0] == 'd' else tz + n
        if nz < g[2] - 0.01 or nz > SAFE_Z:
            print('That would be lower than where it grabbed (or too high), so the arm did not move.')
            continue
        speed = 5
    else:
        if not 0 < n <= 3:
            print('Use a distance between 0 and 3 mm.')
            continue
        dx, dy = MOVES[cmd[0]]
        nx, ny = tx + dx * n, ty + dy * n
        if max(abs(nx - g[0]), abs(ny - g[1])) > 10:
            print('That would be more than 10 mm from where it grabbed, so the arm did not move.')
            continue
        speed = 3
    tx, ty, tz = nx, ny, nz
    arm.move_to_pos([tx, ty, tz, *g[3:]], **at_speed(speed))
    print(f'Moved. Now {tz - g[2]:.1f} mm above where it grabbed.')

if released:
    print('-> up, off the metal part, then high above the bench')
    arm.move_to_pos([tx, ty, min(tz + 40, SAFE_Z), *g[3:]], **at_speed(5))
    arm.move_to_pos([tx, ty, SAFE_Z, *g[3:]], **at_speed(30))
    print('Done. The electrode is back in its stand; the gripper is open, high above it.')
else:
    print('Stopped, still holding the electrode. Run this cell again to carry on.')